In [ ]:
 !pip install bert-score pycocoevalcap
 !pip install -q transformers accelerate qwen-vl-utils \
    nltk rouge-score pillow tqdm pandas matplotlib seaborn

In [ ]:

import torch
import json
import os
from tqdm import tqdm
from pathlib import Path
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer

from google.colab import drive

drive.mount("/content/drive",force_remount=True)
print(os.listdir('/content/drive/MyDrive/data'))

In [ ]:
class Config:
  model_name="Qwen/Qwen2-VL-2B-Instruct"
  device="cpu"
  torch_dtype=torch.float32
  max_new_tokens=256
  json_path   = "/content/drive/MyDrive/data/outputs/drivelm_nusc_val.json"
  image_root  = "/content/drive/MyDrive/data/v1.0-mini"
  num_samples = 100
  output_dir  = "/content/drive/MyDrive/output_qwen/qwen2vl_results"
  os.makedirs(output_dir, exist_ok=True)

In [ ]:
def load_model(config):
  model = Qwen2VLForConditionalGeneration.from_pretrained(
    config.model_name,
    torch_dtype=config.torch_dtype,
    device_map="auto",
    low_cpu_mem_usage=True)
  processor = AutoProcessor.from_pretrained(config.model_name, min_pixels=128*28*28,max_pixels=512*28*28)
  model.eval()
  return model, processor

In [ ]:
import PIL as Image
import numpy as np

def generate_response(model, processor, question, image_path, config):
  content=[]

  image_path = os.path.join(config.image_root,image_path)
  if image_path and os.path.exists(image_path):
    content.append({
        "type":"image",
        "image":image_path,
        "resized_height":280,
        "resized_width":420,
    })
  content.append({"type":"text","text":question})
  messages=[{
      "role":"system",
      "content": "You are an autonomus driving assistant,answer the questions concisely"},
           {"role": "user","content":content},]
  text = processor.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
  image_inputs, _=process_vision_info(messages)

  # if isinstance (image_inputs,list):
  #   image_inputs=Image.fromarray(np.array(image_inputs,dtype=np.uint8))
  inputs=processor(text=[text], images=image_inputs,padding=True, return_tensors="pt")

  with torch.no_grad():
    with torch.inference_mode():
      generated_ids = model.generate(**inputs,max_new_tokens=config.max_new_tokens,
                                     do_sample=False,
                                     num_beams=1,
                                     use_cache=True)
  generated_ids_trimmed=[out_ids[len(in_ids):]
                         for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]

  response=processor.batch_decode(generated_ids_trimmed,skip_special_tokens=True,
                                  clean_up_tokenization_space=False
                                  )[0]
  return response.strip()

In [ ]:

def extract_records(entries):
  QA_CATEGORIES = ["perception", "prediction", "planning"]
  rows = []

  for entry in entries:
      image_rel = entry.get("cam_front_path", "")
      qa_paits = entry.get("qa_paits", {})
      if isinstance(qa_paits, dict):
          for cat in QA_CATEGORIES:
              for qa in qa_paits.get(cat, []):
                  q = (qa.get("Q") or "").strip()
                  a = (qa.get("A") or "").strip()
                  if q and a:
                      rows.append({
                          "question":  q,
                          "answer":    a,
                          "category":  cat,
                          "image" :image_rel,
                      })

  return rows

In [ ]:
import pandas as pd
def run_eval():
  """
  run evaluation loop
  """
  print("started evaluation")
  config=Config()
  model,processor=load_model(config)
  with open(config.json_path,'r') as f:
    entries = json.load(f)
  print(len(entries))
  samples = extract_records(entries)
  if config.num_samples:
    samples=samples[:config.num_samples]


  results =[]
  for idx,sample in enumerate(tqdm(samples, desc="inference")):
    try:

      prediction=generate_response(model,processor, sample['question'],sample.get('image'),config)

      results.append({"question":   sample["question"],
        "reference":  sample["answer"],
        "prediction": prediction,
        "image":      sample["image"],})

    except:
      print(f"error on sample {idx}")
  df_results = pd.DataFrame(results)
  results_csv = os.path.join(config.output_dir, "qwen2vl_predictions_100.csv")
  df_results.to_csv(results_csv, index=False)


In [ ]:
run_eval()

In [ ]:
#
from google.colab import drive
drive.mount("/content/drive",force_remount=True)

import argparse
import re
import json
from collections import defaultdict

import pandas as pd
import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import bert_score




def tokenize(text: str) -> list[str]:
    """Lowercase + simple word tokenization."""
    return nltk.word_tokenize(text.lower())


def extract_numbers(text: str) -> dict[str, int]:
    """
    Extract (object_type → count) pairs from a sentence.
    Handles both word-numbers ("two trucks") and digits ("2 trucks").
    """
    word_map = {
        "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
        "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
        "many": -1, "several": -1, "few": -1,        # -1 = unspecified plural
    }
    counts = {}
    # Match "two trucks" or "2 trucks"
    pattern = r"(\b(?:one|two|three|four|five|six|seven|eight|nine|ten|many|several|few|\d+)\b)\s+([a-z\s]+?)(?=,|and|behind|in front|$)"
    for m in re.finditer(pattern, text.lower()):
        num_str, obj = m.group(1).strip(), m.group(2).strip()
        num = word_map.get(num_str, None)
        if num is None:
            try:
                num = int(num_str)
            except ValueError:
                continue
        counts[obj] = num
    return counts


def object_count_f1(reference: str, prediction: str) -> float:
    """
    Compute F1 over (object_type, count) pairs.
    Treats 'many' / unspecified plurals as a wildcard match.
    """
    ref_counts = extract_numbers(reference)
    pred_counts = extract_numbers(prediction)

    if not ref_counts and not pred_counts:
        return 1.0
    if not ref_counts or not pred_counts:
        return 0.0

    tp = 0
    for obj, ref_n in ref_counts.items():
        if obj in pred_counts:
            pred_n = pred_counts[obj]
            # exact match OR either side is "many" (-1)
            if ref_n == pred_n or ref_n == -1 or pred_n == -1:
                tp += 1

    precision = tp / len(pred_counts) if pred_counts else 0.0
    recall    = tp / len(ref_counts)  if ref_counts  else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_bleu(references: list[str], predictions: list[str]) -> dict:
    """Corpus-level BLEU-1 through BLEU-4."""
    ref_tokens  = [[tokenize(r)] for r in references]
    pred_tokens = [tokenize(p)   for p in predictions]
    smoother = SmoothingFunction().method1
    return {
        f"BLEU-{n}": round(corpus_bleu(ref_tokens, pred_tokens,
                                        weights=tuple([1/n]*n + [0]*(4-n)),
                                        smoothing_function=smoother), 4)
        for n in range(1, 5)
    }


def compute_rouge(references: list[str], predictions: list[str]) -> dict:
    """Average ROUGE-1, ROUGE-2, ROUGE-L F1 scores."""
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    totals = defaultdict(float)
    for ref, pred in zip(references, predictions):
        scores = scorer.score(ref, pred)
        for k, v in scores.items():
            totals[k] += v.fmeasure
    n = len(references)
    return {k.upper(): round(v / n, 4) for k, v in totals.items()}


def compute_meteor(references: list[str], predictions: list[str]) -> dict:
    """Average METEOR score."""
    scores = [
        meteor_score([tokenize(r)], tokenize(p))
        for r, p in zip(references, predictions)
    ]
    return {"METEOR": round(sum(scores) / len(scores), 4)}


def compute_bertscore(references: list[str], predictions: list[str]) -> dict:
    """
    BERTScore F1 (averaged).
    """
    P, R, F1 = bert_score.score(
        predictions, references,
        lang="en",
        rescale_with_baseline=True,
        verbose=False,
    )
    return {
        "BERTScore-P":  round(P.mean().item(),  4),
        "BERTScore-R":  round(R.mean().item(),  4),
        "BERTScore-F1": round(F1.mean().item(), 4),
    }


def compute_object_f1(references: list[str], predictions: list[str]) -> dict:
    """Average object-count F1 across all rows."""
    scores = [object_count_f1(r, p) for r, p in zip(references, predictions)]
    return {"ObjectCount-F1": round(sum(scores) / len(scores), 4)}


# ─────────────────────────────────────────────────────────────
# Per-row scoring (for the output CSV)
# ─────────────────────────────────────────────────────────────

def score_row(ref: str, pred: str) -> dict:
    """Compute all per-sample metrics for a single row."""
    bleu1 = corpus_bleu(
        [[tokenize(ref)]], [tokenize(pred)],
        weights=(1, 0, 0, 0),
        smoothing_function=SmoothingFunction().method1,
    )
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r = rouge.score(ref, pred)
    met = meteor_score([tokenize(ref)], tokenize(pred))
    obj_f1 = object_count_f1(ref, pred)

    return {
        "BLEU-1":         round(bleu1, 4),
        "ROUGE1":         round(r["rouge1"].fmeasure, 4),
        "ROUGE2":         round(r["rouge2"].fmeasure, 4),
        "ROUGEL":         round(r["rougeL"].fmeasure, 4),
        "METEOR":         round(met, 4),
        "ObjectCount-F1": round(obj_f1, 4),
    }


# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────

def evaluate(csv_path: str, output_path: str | None = None,
             category_col: str | None = None,
             filter_category: str | None = None) -> None:

    df = pd.read_csv(csv_path)

    # Validate required columns
    required = {"question", "reference", "prediction"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing columns: {missing}")

    # Optionally filter by category
    if filter_category and category_col and category_col in df.columns:
        df = df[df[category_col].str.lower() == filter_category.lower()]
        print(f"Filtered to '{filter_category}': {len(df)} rows")

    df = df.dropna(subset=["reference", "prediction"])
    refs  = df["reference"].tolist()
    preds = df["prediction"].tolist()

    print(f"\nEvaluating {len(df)} rows …\n")

    # ── Corpus-level metrics ───────────────────────────────────
    all_metrics = {}
    all_metrics.update(compute_bleu(refs, preds))
    all_metrics.update(compute_rouge(refs, preds))
    all_metrics.update(compute_meteor(refs, preds))
    all_metrics.update(compute_object_f1(refs, preds))

    print("── Corpus-level Metrics ──────────────────────────")
    for k, v in all_metrics.items():
        print(f"  {k:<20} {v:.4f}")

    # ── BERTScore (slightly slower, batched) ───────────────────
    print("\nComputing BERTScore (this may take a moment) …")
    bert_metrics = compute_bertscore(refs, preds)
    all_metrics.update(bert_metrics)
    for k, v in bert_metrics.items():
        print(f"  {k:<20} {v:.4f}")

    # ── Per-category breakdown (if column exists) ──────────────
    if category_col and category_col in df.columns:
        print("\n── Per-category Breakdown ────────────────────────")
        for cat, grp in df.groupby(category_col):
            cat_refs  = grp["reference"].tolist()
            cat_preds = grp["prediction"].tolist()
            b = compute_bleu(cat_refs, cat_preds)
            r = compute_rouge(cat_refs, cat_preds)
            m = compute_meteor(cat_refs, cat_preds)
            o = compute_object_f1(cat_refs, cat_preds)
            print(f"\n  [{cat}]  n={len(grp)}")
            for k, v in {**b, **r, **m, **o}.items():
                print(f"    {k:<20} {v:.4f}")

    # ── Per-row scores → output CSV ────────────────────────────
    if output_path:
        row_scores = [score_row(r, p) for r, p in zip(refs, preds)]
        score_df   = pd.DataFrame(row_scores)
        out_df     = df.reset_index(drop=True).join(score_df)
        out_df.to_csv(output_path, index=False)
        print(f"\n Per-row scores saved → {output_path}")

    # ── Summary JSON ───────────────────────────────────────────
    summary_path = (output_path or csv_path).replace(".csv", "_summary.json")
    with open(summary_path, "w") as f:
        json.dump({"n_samples": len(df), "metrics": all_metrics}, f, indent=2)


if __name__ == "__main__":


    evaluate(
        csv_path        = '/content/drive/MyDrive/output_qwen/qwen2vl_results/qwen2vl_predictions_100.csv',
        output_path     = '/content/drive/MyDrive/output_qwen/qwen2vl_results/metrics.csv',
        category_col    = None,
        filter_category = None
    )



